In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [2]:
# ---------------------------------------------------------
# 1. Datos de Entrada y Parámetros
# ---------------------------------------------------------
asset_names = ['Stocks', 'Bonds', 'Real E.']

E = np.array([0.065, 0.040, 0.050])
sigma = np.array([0.16, 0.07, 0.12])

Omega = np.array([
    [0.02560, 0.00392, 0.00384],
    [0.00392, 0.00490, 0.00168],
    [0.00384, 0.00168, 0.01440]
])

ones = np.ones(len(E))
Omega_inv = np.linalg.inv(Omega)

# Tasa libre de riesgo (rf) y Coeficiente de Aversión al Riesgo (gamma)
rf = 0.020      # 2.0%
gamma = 3.5     # Coeficiente gamma del inversionista

In [3]:
# ---------------------------------------------------------
# 2. Portafolio de Tangencia (T)
# ---------------------------------------------------------
excess_returns = E - rf * ones
w_tangency_unscaled = Omega_inv @ excess_returns
w_tangency = w_tangency_unscaled / np.sum(w_tangency_unscaled)

E_T = float(w_tangency.T @ E)
var_T = float(w_tangency.T @ Omega @ w_tangency)
sigma_T = np.sqrt(var_T)
sharpe = (E_T - rf) / sigma_T

In [4]:
# ---------------------------------------------------------
# 3. Portafolio Óptimo (w*)
# ---------------------------------------------------------
# Proporción a invertir en el Portafolio de Tangencia:
w_star = (1 / gamma) * ((E_T - rf) / var_T)

# Rendimiento y Volatilidad del Portafolio Óptimo Combinado
E_opt = rf + w_star * (E_T - rf)
sigma_opt = w_star * sigma_T

# Nivel de Utilidad Máximo del Inversionista
U_max = E_opt - 0.5 * gamma * (sigma_opt ** 2)

# Pesos absolutos sobre los activos individuales
w_assets_opt = w_star * w_tangency

In [5]:
# ---------------------------------------------------------
# 4. Curvas (CML, Frontera y Curva de Indiferencia)
# ---------------------------------------------------------
# CML
sigma_cml = np.linspace(0, 0.22, 200)
E_cml = rf + sharpe * sigma_cml

# Curva de Indiferencia Tangente: U = E - (gamma/2)*sigma^2  =>  E = U + (gamma/2)*sigma^2
sigma_indiff = np.linspace(0, 0.22, 200)
E_indiff = U_max + 0.5 * gamma * (sigma_indiff ** 2)

# Frontera de Markowitz
a11 = float(E.T @ Omega_inv @ E)
a12 = float(ones.T @ Omega_inv @ E)
a22 = float(ones.T @ Omega_inv @ ones)
A = np.array([[a11, a12], [a12, a22]])
delta = np.linalg.det(A)

mu_vals = np.linspace(0.01, 0.11, 500)
sigma_p = np.sqrt((a22 / delta) * (mu_vals - (a12 / a22))**2 + (1 / a22))

In [6]:
# ---------------------------------------------------------
# Impresión de Resultados
# ---------------------------------------------------------
print("=" * 60)
print("        OPTIMIZACIÓN DE UTILIDAD Y PORTAFOLIO ÓPTIMO       ")
print("=" * 60)
print(f"Coeficiente de Aversión al Riesgo (γ) : {gamma}")
print(f"Tasa Libre de Riesgo (rf)             : {rf:.2%}")
print(f"Retorno Tangencia (E_T)               : {E_T:.2%}")
print(f"Varianza Tangencia (σ_T^2)            : {var_T:.6f}")
print("-" * 60)
print(f"Proporción en Tangencia (w*)          : {w_star:.4f} ({w_star:.2%})")
print(f"Proporción en Activo Seguro (1 - w*)  : {(1 - w_star):.4f} ({(1 - w_star):.2%})")
print("-" * 60)
print(f"Retorno Portafolio Óptimo (E*)        : {E_opt:.2%}")
print(f"Volatilidad Portafolio Óptimo (σ*)    : {sigma_opt:.2%}")
print(f"Nivel de Utilidad Máximo (U*)         : {U_max:.4f}")
print("=" * 60)
print("Asignación de Capital Neta en Activos Individuales:")
for name, w in zip(asset_names, w_assets_opt):
    print(f"  - {name:10s}: {w:7.2%}")
print("=" * 60)

        OPTIMIZACIÓN DE UTILIDAD Y PORTAFOLIO ÓPTIMO       
Coeficiente de Aversión al Riesgo (γ) : 3.5
Tasa Libre de Riesgo (rf)             : 2.00%
Retorno Tangencia (E_T)               : 4.81%
Varianza Tangencia (σ_T^2)            : 0.005335
------------------------------------------------------------
Proporción en Tangencia (w*)          : 1.5067 (150.67%)
Proporción en Activo Seguro (1 - w*)  : -0.5067 (-50.67%)
------------------------------------------------------------
Retorno Portafolio Óptimo (E*)        : 6.24%
Volatilidad Portafolio Óptimo (σ*)    : 11.01%
Nivel de Utilidad Máximo (U*)         : 0.0412
Asignación de Capital Neta en Activos Individuales:
  - Stocks    :  32.22%
  - Bonds     :  76.44%
  - Real E.   :  42.01%


In [10]:
# ---------------------------------------------------------
# 5. Visualización con Plotly
# ---------------------------------------------------------
fig = go.Figure()

# Frontera de Mínima Varianza (completa)
fig.add_trace(go.Scatter(
    x=sigma_p, y=mu_vals,
    mode='lines',
    name='Frontera de Mínima Varianza',
    line=dict(color='gray', dash='dash', width=1.5),
    hovertemplate='σ: %{x:.2%}<br>E: %{y:.2%}'
))

# Frontera Eficiente de Markowitz
mask_eff = mu_vals >= (a12 / a22)
fig.add_trace(go.Scatter(
    x=sigma_p[mask_eff], y=mu_vals[mask_eff],
    mode='lines', name='Frontera Eficiente (Solo Activos Riesgosos)',
    line=dict(color='#00CC96', width=3),
))

# Capital Market Line (CML)
fig.add_trace(go.Scatter(
    x=sigma_cml, y=E_cml,
    mode='lines', name='Capital Market Line (CML)',
    line=dict(color='#FFD700', width=2, dash='dot'),
))

# Curva de Indiferencia
fig.add_trace(go.Scatter(
    x=sigma_indiff, y=E_indiff,
    mode='lines', name=f'Curva de Indiferencia (γ={gamma})',
    line=dict(color='red', width=3),
    hovertemplate='σ: %{x:.2%}<br>E: %{y:.2%}'
))

# Activo Libre de Riesgo
fig.add_trace(go.Scatter(
    x=[0], y=[rf], mode='markers+text', name='Activo Libre de Riesgo (rf)',
    text=['rf (2%)'], textposition='top right',
    marker=dict(size=9, color='white', line=dict(width=2, color='orange'))
))

# Portafolio de Tangencia
fig.add_trace(go.Scatter(
    x=[sigma_T], y=[E_T], mode='markers+text', name='Portafolio de Tangencia (T)',
    text=['MVE Portfolio'], textposition='top left',
    marker=dict(size=12, color='white', symbol='star', line=dict(width=0, color='white')),
))

# optimnal portfolio
fig.add_trace(go.Scatter(
    x=[sigma_opt], y=[E_opt], mode='markers+text', name='Inversor\'s Optimal Portfolio',
    text=['Optimal Portfolio'], textposition='bottom right',
    marker=dict(size=12, color='white'),
    hovertemplate=f'<b>Portafolio Óptimo</b><br>σ: %{{x:.2%}}<br>E: %{{y:.2%}}<br>w* en T: {w_star:.2%}'
))

# Layout
fig.update_layout(
    title='<b>Optimización de Utilidad, CML y Portafolio Óptimo</b>',
    xaxis_title='Riesgo / Volatilidad (σ)',
    yaxis_title='Retorno Esperado (E)',
    xaxis=dict(tickformat='.1%', range=[-0.005, 0.20]),
    yaxis=dict(tickformat='.1%', range=[0.01, 0.095]),
    template='plotly_dark',
    hovermode='closest',
    width=900,
    height=600
)

fig.show()